# 04. Evaluation

Load cac model da train, predict tren test set, inverse scale ve `traffic_volume` goc, tinh MAE/RMSE/MAPE/R2, so sanh 3 mo hinh va ve bieu do ket qua.

## 1. Import thu vien

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='Set2')

## 2. Load du lieu test va scaler

In [ ]:
candidate_processed_dirs = [
    Path.cwd() / '..' / 'data' / 'processed',
    Path.cwd() / 'data' / 'processed',
    Path.cwd() / 'time_series _' / 'data' / 'processed',
]

processed_dir = next((path.resolve() for path in candidate_processed_dirs if (path / 'feature_metadata.json').exists()), None)
if processed_dir is None:
    raise FileNotFoundError('Khong tim thay data/processed. Hay chay notebook 02_feature_engineering.ipynb truoc.')

project_dir = processed_dir.parents[1]
model_dir = project_dir / 'results' / 'models'
evaluation_dir = project_dir / 'results' / 'evaluation'
figure_dir = project_dir / 'figures' / 'evaluation'
evaluation_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

with open(processed_dir / 'feature_metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
target_col = metadata['target_col']
seq_len = metadata['seq_len']
pred_len = metadata['pred_len']

target_scaler = joblib.load(processed_dir / 'target_scaler.pkl')
test_npz = np.load(processed_dir / 'test_windows.npz')

X_test_seq = test_npz['X'].astype(np.float32)
y_test_scaled = test_npz['y'].astype(np.float32)
target_start = pd.to_datetime(test_npz['target_start'])
target_end = pd.to_datetime(test_npz['target_end'])
X_test_tab = X_test_seq[:, -1, :]

print(f'Processed dir: {processed_dir}')
print(f'Model dir: {model_dir}')
print(f'X_test_seq: {X_test_seq.shape}, y_test: {y_test_scaled.shape}')
print(f'Target window: {target_start.min()} -> {target_end.max()}')

## 3. Ham inverse scale va metrics

In [ ]:
def inverse_target(y_scaled):
    y_scaled = np.asarray(y_scaled)
    y_2d = y_scaled.reshape(-1, 1)
    return target_scaler.inverse_transform(y_2d).reshape(y_scaled.shape)


def regression_metrics_original_scale(y_true, y_pred):
    y_true_flat = np.asarray(y_true).reshape(-1)
    y_pred_flat = np.asarray(y_pred).reshape(-1)
    nonzero_mask = y_true_flat != 0
    rmse = mean_squared_error(y_true_flat, y_pred_flat) ** 0.5
    mape = np.mean(np.abs((y_true_flat[nonzero_mask] - y_pred_flat[nonzero_mask]) / y_true_flat[nonzero_mask])) * 100
    return {
        'MAE': mean_absolute_error(y_true_flat, y_pred_flat),
        'RMSE': rmse,
        'MAPE (%)': mape,
        'R2': r2_score(y_true_flat, y_pred_flat),
    }


def horizon_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred), axis=0)


y_test = inverse_target(y_test_scaled)
print('Da inverse scale y_test ve traffic_volume goc:', y_test.shape)

## 4. Load model da train

In [ ]:
class ITransformer(nn.Module):
    def __init__(self, seq_len, pred_len, n_features, d_model=128, n_heads=8, num_layers=3, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.n_features = n_features
        self.value_embedding = nn.Linear(seq_len, d_model)
        self.variable_embedding = nn.Parameter(torch.randn(1, n_features, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, pred_len),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.value_embedding(x) + self.variable_embedding
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.head(x)


def load_torch_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


linear_model = joblib.load(model_dir / 'linear_regression.pkl')
xgb_model = joblib.load(model_dir / 'xgboost_multioutput.pkl')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = load_torch_checkpoint(model_dir / 'itransformer.pt', device)
config = checkpoint.get('model_config', {})
itransformer_model = ITransformer(
    seq_len=config.get('seq_len', seq_len),
    pred_len=config.get('pred_len', pred_len),
    n_features=config.get('n_features', len(feature_cols)),
    d_model=config.get('d_model', 64),
    n_heads=config.get('n_heads', 4),
    num_layers=config.get('num_layers', 2),
    dropout=config.get('dropout', 0.1),
).to(device)
itransformer_model.load_state_dict(checkpoint['model_state_dict'])
itransformer_model.eval()

print('Da load 3 model: Linear Regression, XGBoost, iTransformer')
print(f'Device cho iTransformer: {device}')

## 5. Predict tren test set

In [ ]:
def predict_itransformer(model, X, batch_size=256):
    dataset = TensorDataset(torch.from_numpy(X))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    preds = []
    model.eval()
    with torch.no_grad():
        for (X_batch,) in loader:
            X_batch = X_batch.to(device)
            preds.append(model(X_batch).cpu().numpy())
    return np.concatenate(preds, axis=0)


predictions_scaled = {
    'Linear Regression': linear_model.predict(X_test_tab),
    'XGBoost': xgb_model.predict(X_test_tab),
    'iTransformer': predict_itransformer(itransformer_model, X_test_seq),
}

predictions = {name: inverse_target(pred) for name, pred in predictions_scaled.items()}

for name, pred in predictions.items():
    print(f'{name}: scaled={predictions_scaled[name].shape}, original_scale={pred.shape}')

## 6. Tinh MAE, RMSE, MAPE, R2 va so sanh 3 mo hinh

In [ ]:
metrics_rows = []
for model_name, y_pred in predictions.items():
    row = {'Model': model_name}
    row.update(regression_metrics_original_scale(y_test, y_pred))
    metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows).sort_values('RMSE').reset_index(drop=True)
metrics_df.to_csv(evaluation_dir / 'test_metrics.csv', index=False)
display(metrics_df)

best_model_name = metrics_df.loc[0, 'Model']
print(f'Mo hinh tot nhat theo RMSE tren test set: {best_model_name}')

In [ ]:
horizon_rows = []
for model_name, y_pred in predictions.items():
    for horizon_idx, mae in enumerate(horizon_mae(y_test, y_pred), start=1):
        horizon_rows.append({
            'Model': model_name,
            'Horizon': horizon_idx,
            'MAE': mae,
        })

horizon_metrics_df = pd.DataFrame(horizon_rows)
horizon_metrics_df.to_csv(evaluation_dir / 'horizon_mae.csv', index=False)
display(horizon_metrics_df.head())

## 7. Ve bieu do ket qua

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metric_names = ['MAE', 'RMSE', 'MAPE (%)', 'R2']

for ax, metric in zip(axes.ravel(), metric_names):
    sns.barplot(data=metrics_df, x='Model', y=metric, ax=ax)
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

fig.suptitle('So sanh metrics tren test set', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(figure_dir / 'test_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(13, 5))
sns.lineplot(data=horizon_metrics_df, x='Horizon', y='MAE', hue='Model', marker='o')
plt.title('MAE theo tung buoc du bao 1-24 gio')
plt.xlabel('Forecast horizon (gio)')
plt.ylabel('MAE traffic_volume')
plt.xticks(range(1, pred_len + 1))
plt.tight_layout()
plt.savefig(figure_dir / 'horizon_mae.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plot_points = min(24 * 7, len(y_test))
time_index = target_start[:plot_points]

plt.figure(figsize=(15, 6))
plt.plot(time_index, y_test[:plot_points, 0], label='Actual', color='black', linewidth=2)
for model_name, y_pred in predictions.items():
    plt.plot(time_index, y_pred[:plot_points, 0], label=model_name, alpha=0.85)

plt.title('Actual vs Prediction cho horizon +1 gio')
plt.xlabel('Thoi gian')
plt.ylabel('traffic_volume')
plt.legend()
plt.tight_layout()
plt.savefig(figure_dir / 'actual_vs_prediction_horizon_1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
scatter_sample = min(5000, y_test.size)
rng = np.random.default_rng(42)
sample_idx = rng.choice(y_test.size, size=scatter_sample, replace=False)
actual_flat = y_test.reshape(-1)[sample_idx]
min_value = min(actual_flat.min(), *(pred.reshape(-1)[sample_idx].min() for pred in predictions.values()))
max_value = max(actual_flat.max(), *(pred.reshape(-1)[sample_idx].max() for pred in predictions.values()))

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)
for ax, (model_name, y_pred) in zip(axes, predictions.items()):
    pred_flat = y_pred.reshape(-1)[sample_idx]
    ax.scatter(actual_flat, pred_flat, s=8, alpha=0.25)
    ax.plot([min_value, max_value], [min_value, max_value], color='red', linestyle='--', linewidth=1)
    ax.set_title(model_name)
    ax.set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
fig.suptitle('Scatter Actual vs Predicted tren test set', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(figure_dir / 'actual_vs_predicted_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
prediction_rows = []
for i in range(len(y_test)):
    for horizon_idx in range(pred_len):
        row = {
            'target_start': target_start[i],
            'horizon': horizon_idx + 1,
            'actual_traffic_volume': y_test[i, horizon_idx],
        }
        for model_name, y_pred in predictions.items():
            row[f'{model_name}_prediction'] = y_pred[i, horizon_idx]
        prediction_rows.append(row)

prediction_df = pd.DataFrame(prediction_rows)
prediction_df.to_csv(evaluation_dir / 'test_predictions_original_scale.csv', index=False)
display(prediction_df.head())

print(f'Da luu metrics tai: {evaluation_dir / "test_metrics.csv"}')
print(f'Da luu predictions tai: {evaluation_dir / "test_predictions_original_scale.csv"}')
print(f'Da luu figures tai: {figure_dir}')